# PUDM — DDPM Training & Evaluation

This notebook trains and evaluates **PUDM** (Point Cloud Upsampling via Denoising Diffusion Model) using the **DDPM** strategy (baseline from CVPR 2024 paper).

For the Flow Matching variant, see `pudm_flow_matching.ipynb`.

> **Prerequisite:** Run `config.ipynb` first to clone the repo, install dependencies, and compile CUDA extensions.

## 1. Setup

Mount Drive and load the repo (already cloned by `config.ipynb`).

In [ ]:
import os, sys, shutil, glob as _glob

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Repo lives on Drive (cloned by config.ipynb)
REPO_DIR = '/content/drive/MyDrive/MVA/pointcloud/pudm_extension'
assert os.path.isdir(REPO_DIR), f'Repo not found at {REPO_DIR} — run config.ipynb first'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install Python dependencies
!pip install -q -r requirements.txt

import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Restore cached CUDA extensions (compiled by config.ipynb)
_tag = f'py{sys.version_info.major}{sys.version_info.minor}_pt{torch.__version__.split("+")[0]}_cu{torch.version.cuda.replace(".", "")}'
CACHE_DIR = os.path.join(REPO_DIR, 'cache', _tag)

POINTOPS_SRC = os.path.join(REPO_DIR, 'src', 'ops', 'pointops')
PNET2_PKG    = os.path.join(REPO_DIR, 'src', 'ops', 'pointnet2_ops')

# pointops
for so in _glob.glob(os.path.join(CACHE_DIR, 'pointops', 'pointops_cuda*.so')):
    shutil.copy2(so, POINTOPS_SRC)
    print(f'✓ pointops: {os.path.basename(so)}')

# pointnet2_ops
for so in _glob.glob(os.path.join(CACHE_DIR, 'pointnet2_ops', '_ext*.so')):
    shutil.copy2(so, PNET2_PKG)
    print(f'✓ pointnet2_ops: {os.path.basename(so)}')

if not _glob.glob(os.path.join(CACHE_DIR, '**', '*.so'), recursive=True):
    raise FileNotFoundError(f'No cached .so files in {CACHE_DIR} — run config.ipynb first')

# Verify imports
from src.generative import get_strategy, STRATEGIES
from src.utils.config import load_config
from src.utils.misc import set_seed

print(f'\nStrategies: {list(STRATEGIES.keys())}')
print('All imports OK!')

## 2. Configuration

Choose your dataset, strategy, and data paths.

In [ ]:
# ============ EDIT THESE ============
DATASET = 'PU1K'                    # 'PU1K' or 'PUGAN'
STRATEGY_NAME = 'ddpm'              # fixed for this notebook
R = 4                               # upsampling rate
GAMMA = 0.5                         # condition interpolation
SEED = 42
# ====================================

import shutil, time as _time
from src.utils.config import get_strategy_config

# Copy dataset from Drive to local SSD for fast I/O
DATA_DRIVE = os.path.join(REPO_DIR, 'data', DATASET)
DATA_DIR   = os.path.join('/content/data', DATASET)

if os.path.isdir(DATA_DIR):
    print(f'✓ {DATASET}: already on local disk')
elif os.path.isdir(DATA_DRIVE):
    t0 = _time.time()
    print(f'{DATASET}: copying from Drive to local SSD ...')
    shutil.copytree(DATA_DRIVE, DATA_DIR)
    print(f'✓ {DATASET}: copied in {_time.time()-t0:.1f}s')
else:
    raise FileNotFoundError(
        f'Dataset not found at {DATA_DRIVE}\n'
        f'Place your {DATASET} data in {DATA_DRIVE} (see config.ipynb).'
    )

CHECKPOINT_DIR = os.path.join(REPO_DIR, 'checkpoints', DATASET.lower(), STRATEGY_NAME)
OUTPUT_DIR = os.path.join(REPO_DIR, 'outputs', DATASET.lower())

set_seed(SEED)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load config
config_path = os.path.join(REPO_DIR, 'configs', f'{DATASET}.json')
config = load_config(config_path)

# Override data directory in config (point at local copy)
dataset_key = 'pu1k_dataset_config' if DATASET == 'PU1K' else 'pugan_dataset_config'
config[dataset_key]['data_dir'] = DATA_DIR

# Get strategy and its config section
strategy = get_strategy(STRATEGY_NAME)
strategy_config = get_strategy_config(config, STRATEGY_NAME)
hyperparams = strategy.compute_hyperparams(**strategy_config)

# Move hyperparams to GPU
for key in hyperparams:
    if key != 'T' and isinstance(hyperparams[key], torch.Tensor):
        hyperparams[key] = hyperparams[key].cuda()

print(f'Dataset: {DATASET}')
print(f'Strategy: {strategy.name}')
print(f'Data dir: {DATA_DIR} (local SSD)')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'T = {hyperparams["T"]}')

## 3. Training

Train the model with your chosen strategy. Skip this cell if you already have a checkpoint.

In [ ]:
import time
import torch.nn as nn
from torch.amp import GradScaler, autocast
from src.data.dataset import get_dataloader
from src.models.pointnet2_with_pcld_condition import PointNet2CloudCondition

# Confirm actual GPU right before training (device name can be stale)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# Training params
train_config = config['train_config']
pointnet_config = config['pointnet_config']
trainset_config = config[dataset_key]

# Auto-scale batch size & workers for the current GPU
gpu_name = torch.cuda.get_device_name(0).lower()
if 'a100' in gpu_name or 'h100' in gpu_name or 'a10' in gpu_name:
    trainset_config['batch_size'] = 128
    trainset_config['num_workers'] = 8
    print(f'High-end GPU detected → batch_size=128, num_workers=8')
elif 'v100' in gpu_name or 'l4' in gpu_name:
    trainset_config['batch_size'] = 64
    trainset_config['num_workers'] = 4
    print(f'Mid-tier GPU detected → batch_size=64, num_workers=4')
else:
    print(f'Using config defaults → batch_size={trainset_config["batch_size"]}, num_workers={trainset_config["num_workers"]}')

N_EPOCHS = 200                      # original paper uses 2000; 200 is enough for comparison
LR = train_config.get('learning_rate', 2e-4)
EPOCHS_PER_CKPT = train_config.get('epochs_per_ckpt', 10)
LOG_EVERY = train_config.get('iters_per_logging', 50)

# DataLoader
trainloader = get_dataloader(trainset_config)
print(f'Training samples: {len(trainloader.dataset)}')
print(f'Batches per epoch: {len(trainloader)}')

# Model
net = PointNet2CloudCondition(pointnet_config).cuda()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
loss_fn = nn.MSELoss()
scaler = GradScaler('cuda')

# Resume from checkpoint if available
start_epoch = 0
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')]) if os.path.exists(CHECKPOINT_DIR) else []
if ckpt_files:
    latest = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
    ckpt = torch.load(latest, map_location='cpu')
    net.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'Resumed from {latest} at epoch {start_epoch}')

print(f'\nTraining {strategy.name} for {N_EPOCHS} epochs (starting from {start_epoch})...')

In [ ]:
# Training loop (mixed precision)
net.train()
n_iter = start_epoch * len(trainloader)
n_iters_total = N_EPOCHS * len(trainloader)
time0 = time.time()

for epoch in range(start_epoch + 1, N_EPOCHS + 1):
    epoch_loss = 0.0
    for data in trainloader:
        label = data['label'].cuda()
        X = data['complete'].cuda()
        condition = data['partial'].cuda()

        optimizer.zero_grad()
        with autocast('cuda'):
            loss = strategy.training_loss(net, loss_fn, X, hyperparams,
                                         label=label, condition=condition)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_iter += 1

        if n_iter % LOG_EVERY == 0:
            print(f'  epoch {epoch}/{N_EPOCHS}  iter {n_iter}/{n_iters_total}  loss: {loss.item():.6f}')

    avg_loss = epoch_loss / len(trainloader)

    if epoch % EPOCHS_PER_CKPT == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'pointnet_ckpt_{epoch}_{avg_loss:.6f}.pkl')
        torch.save({
            'epoch': epoch,
            'iter': n_iter,
            'model_state_dict': net.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'training_time_seconds': int(time.time() - time0),
            'strategy': strategy.name,
        }, ckpt_path)
        print(f'  -> Saved checkpoint: {ckpt_path}')

print(f'\nTraining complete. Total time: {(time.time() - time0)/60:.1f} min')

## 4. Sampling & Evaluation

Generate dense point clouds from the test set and compute metrics.

In [ ]:
from src.scripts.eval import evaluate

# Choose checkpoint
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')])
if ckpt_files:
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
else:
    raise FileNotFoundError(f'No checkpoints in {CHECKPOINT_DIR}')

print(f'Using checkpoint: {CKPT_PATH}')

# Load model
net_eval = PointNet2CloudCondition(pointnet_config).cuda()
ckpt = torch.load(CKPT_PATH, map_location='cpu')
net_eval.load_state_dict(ckpt['model_state_dict'], strict=False)
net_eval.eval()

# Test dataloader
test_config = config[dataset_key].copy()
test_config['batch_size'] = 14
test_config['eval_batch_size'] = 14
testloader = get_dataloader(test_config, phase='test')
print(f'Test samples: {len(testloader.dataset)}')

STEP = 30  # DDIM steps (for DDPM) or Euler steps (for FM)
SAVE_DIR = os.path.join(OUTPUT_DIR, 'samples')
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
with torch.no_grad():
    CD, HD, P2F, meta, metrics = evaluate(
        net=net_eval,
        testloader=testloader,
        strategy=strategy,
        hyperparams=hyperparams,
        print_every_n_steps=strategy_config['T'] // 5,
        scale=config[dataset_key].get('scale', 1),
        compute_cd=True,
        return_all_metrics=True,
        R=R,
        npoints=config[dataset_key]['npoints'],
        T=strategy_config['T'],
        step=STEP,
        save_dir=SAVE_DIR,
        gamma=GAMMA,
    )

print(f'\n{"="*50}')
print(f'Strategy: {strategy.name}')
print(f'Upsampling: {config[dataset_key]["npoints"]} -> {config[dataset_key]["npoints"] * R}')
print(f'Chamfer Distance (CD): {CD:.8f}')
print(f'Hausdorff Distance (HD): {HD:.8f}')
print(f'Point-to-Face (P2F): {P2F:.8f}')
print(f'{"="*50}')

## 5. Single-File Example

Upsample a single `.xyz` point cloud and visualize the result.

In [ ]:
import numpy as np
import open3d
from src.utils.pc_utils import pc_normalize, numpy_to_pc

# Path to a single .xyz file
EXAMPLE_FILE = '/content/drive/MyDrive/MVA/pointcloud/data/example/pig.xyz'  # EDIT THIS

if os.path.exists(EXAMPLE_FILE):
    pc = open3d.io.read_point_cloud(EXAMPLE_FILE)
    pts = np.asarray(pc.points, dtype=np.float32)
    pts = pc_normalize(pts)
    condition = torch.from_numpy(pts).unsqueeze(0).cuda()
    npoints = condition.shape[1]
    label = torch.ones(1, dtype=torch.long).cuda() * (R - 1)

    net_eval.eval()
    net_eval.reset_cond_features()

    with torch.no_grad():
        gen, cond_pre, z = strategy.sample_ddim(
            net=net_eval, size=(1, npoints * R, 3),
            hyperparams=hyperparams, label=label,
            condition=condition, R=R, gamma=GAMMA, step=STEP,
        )

    gen_np = gen[0].cpu().numpy()
    print(f'Input: {npoints} points -> Generated: {gen_np.shape[0]} points')

    # Save result
    out_path = os.path.join(OUTPUT_DIR, 'example_output.xyz')
    open3d.io.write_point_cloud(out_path, numpy_to_pc(gen_np))
    print(f'Saved to: {out_path}')
else:
    print(f'Example file not found: {EXAMPLE_FILE}')
    print('Edit EXAMPLE_FILE path above, or skip this cell.')